# 02 — ETL paralelo de datos abiertos de COVID-19 e influenza (México)

**Responsable:** Integrante 2 — procesamiento paralelo y Docker.

## Objetivo

Implementar un ETL paralelo utilizando Python y ProcessPoolExecutor.
Se aplicarán las mismas reglas de limpieza que en la versión secuencial
para comparar correctamente los tiempos de ejecución.

### 1. Importar librerías

In [2]:

import os
import time
import numpy as np
import pandas as pd

from concurrent.futures import ProcessPoolExecutor
from time import perf_counter

print("CPU disponibles:", os.cpu_count())
print("Librerías cargadas correctamente")

CPU disponibles: 20
Librerías cargadas correctamente


### 2. Cargar dataset original

In [3]:
# 2. Cargar dataset original

RUTA_CSV = "datos/COVID19MEXICO.csv"

df_original = pd.read_csv(
    RUTA_CSV,
    encoding="utf-8-sig",
    low_memory=False
)

print("Filas originales:", len(df_original))
print("Columnas originales:", len(df_original.columns))

display(df_original.head())

Filas originales: 108733
Columnas originales: 42


,FECHA_ACTUALIZACION,ID_REGISTRO,ORIGEN,SECTOR,ENTIDAD_UM,SEXO,ENTIDAD_NAC,ENTIDAD_RES,MUNICIPIO_RES,TIPO_PACIENTE,...,RESULTADO_PCR,RESULTADO_PCR_COINFECCION,TOMA_MUESTRA_ANTIGENO,RESULTADO_ANTIGENO,CLASIFICACION_FINAL_COVID,CLASIFICACION_FINAL_FLU,MIGRANTE,PAIS_NACIONALIDAD,PAIS_ORIGEN,UCI
0,2026-11-02,g829cab,1,12,11,2,11,11,15,1,...,997,997,2,97,6,6,99,México,97,97
1,2026-11-02,gc633cc,1,8,9,2,7,9,12,1,...,997,997,2,97,6,6,99,México,97,97
2,2026-11-02,gad762a,1,4,30,2,30,30,87,2,...,3,998,2,97,7,3,99,México,97,2
3,2026-11-02,gd88222,1,8,9,2,9,9,10,1,...,997,997,2,97,6,6,99,México,97,97
4,2026-11-02,g8447ee,1,4,9,2,9,9,6,2,...,997,997,2,97,6,6,99,México,97,2


### 3. Función de transformación para cada bloque

In [4]:
# 3. Función de transformación para cada bloque

COLUMNAS_FECHA = [
    "FECHA_ACTUALIZACION",
    "FECHA_INGRESO",
    "FECHA_SINTOMAS",
    "FECHA_DEF",
]

def transformar_bloque(tarea):
    bloque, columnas_fecha = tarea
    datos = bloque.copy()

    # Convertir fechas
    for columna in columnas_fecha:
        if columna in datos.columns:
            fechas = datos[columna].astype("string").str.strip()
            fechas = fechas.replace({
                "9999-99-99": pd.NA,
                "": pd.NA
            })
            datos[columna] = pd.to_datetime(
                fechas,
                errors="coerce"
            )

    # Validar edad
    edades_invalidas = 0

    if "EDAD" in datos.columns:
        datos["EDAD"] = pd.to_numeric(
            datos["EDAD"],
            errors="coerce"
        )

        mascara = (
            datos["EDAD"].notna()
            & ~datos["EDAD"].between(0, 120)
        )

        edades_invalidas = int(mascara.sum())
        datos.loc[mascara, "EDAD"] = np.nan

    # Limpiar espacios en columnas de texto
    columnas_texto = datos.select_dtypes(
        include=["object", "string"]
    ).columns

    for columna in columnas_texto:
        datos[columna] = (
            datos[columna].astype("string").str.strip()
        )

    return datos, edades_invalidas


print("Función de transformación paralela preparada.")

Función de transformación paralela preparada.


### 4. Procesamiento paralelo 

In [7]:

# 4. Procesamiento paralelo 

def ejecutar_etl_paralelo(df, numero_procesos=None):

    inicio = perf_counter()

    datos = df.copy()

    # Normalizar encabezados
    datos.columns = datos.columns.str.strip().str.upper()

    # Eliminar duplicados exactos globalmente
    duplicados_exactos = int(datos.duplicated().sum())
    datos = datos.drop_duplicates()

    # Eliminar ID_REGISTRO repetidos globalmente
    duplicados_id = 0

    if "ID_REGISTRO" in datos.columns:
        duplicados_id = int(
            datos["ID_REGISTRO"].duplicated().sum()
        )
        datos = datos.drop_duplicates(
            subset=["ID_REGISTRO"],
            keep="first"
        )

    # Determinar número de procesos
    if numero_procesos is None:
        numero_procesos = min(4, os.cpu_count() or 1)

    numero_procesos = max(1, numero_procesos)

    # Dividir los índices, conservando DataFrames de pandas
    indices = np.array_split(
        np.arange(len(datos)),
        numero_procesos
    )

    bloques = [
        datos.iloc[indice].copy()
        for indice in indices
        if len(indice) > 0
    ]

    tareas = [
        (bloque, COLUMNAS_FECHA)
        for bloque in bloques
    ]

    # Ejecutar los bloques en paralelo
    with ProcessPoolExecutor(
        max_workers=numero_procesos
    ) as executor:

        resultados = list(
            executor.map(transformar_bloque, tareas)
        )

    # Unir resultados
    df_resultado = pd.concat(
        [resultado[0] for resultado in resultados]
    )

    edades_invalidas = sum(
        resultado[1] for resultado in resultados
    )

    tiempo = perf_counter() - inicio

    auditoria = {
        "filas_entrada": len(df),
        "duplicados_exactos_eliminados": duplicados_exactos,
        "duplicados_id_eliminados": duplicados_id,
        "edades_invalidas_convertidas_a_nulo": edades_invalidas,
        "filas_salida": len(df_resultado),
        "procesos_utilizados": len(bloques),
        "tiempo_segundos": tiempo
    }

    return df_resultado, auditoria


In [8]:
df_paralelo, auditoria_paralela = ejecutar_etl_paralelo(
    df_original,
    numero_procesos=4
)

display(pd.DataFrame([auditoria_paralela]))

,filas_entrada,duplicados_exactos_eliminados,duplicados_id_eliminados,edades_invalidas_convertidas_a_nulo,filas_salida,procesos_utilizados,tiempo_segundos
0,108733,0,0,0,108733,4,1.051869


In [10]:

# Función ETL secuencial de referencia
# Adaptada del notebook de Persona 1

def transformar_datos(df):
    datos = df.copy()
    filas_iniciales = len(datos)

    # 1. Normalizar encabezados
    datos.columns = datos.columns.str.strip().str.upper()

    # 2. Eliminar duplicados exactos
    duplicados_exactos = int(datos.duplicated().sum())
    datos = datos.drop_duplicates()

    # 3. Eliminar ID_REGISTRO repetidos
    duplicados_id = 0
    if "ID_REGISTRO" in datos.columns:
        duplicados_id = int(datos["ID_REGISTRO"].duplicated().sum())
        datos = datos.drop_duplicates(
            subset=["ID_REGISTRO"],
            keep="first"
        )

    # 4. Transformar fechas
    for columna in COLUMNAS_FECHA:
        if columna in datos.columns:
            fechas = datos[columna].astype("string").str.strip()
            fechas = fechas.replace({
                "9999-99-99": pd.NA,
                "": pd.NA
            })
            datos[columna] = pd.to_datetime(
                fechas,
                errors="coerce"
            )

    # 5. Validar edades
    edades_invalidas = 0
    if "EDAD" in datos.columns:
        datos["EDAD"] = pd.to_numeric(
            datos["EDAD"],
            errors="coerce"
        )

        mascara = (
            datos["EDAD"].notna()
            & ~datos["EDAD"].between(0, 120)
        )

        edades_invalidas = int(mascara.sum())
        datos.loc[mascara, "EDAD"] = np.nan

    # 6. Limpiar espacios en textos
    columnas_texto = datos.select_dtypes(
        include=["object", "string"]
    ).columns

    for columna in columnas_texto:
        datos[columna] = (
            datos[columna].astype("string").str.strip()
        )

    auditoria = {
        "filas_entrada": filas_iniciales,
        "duplicados_exactos_eliminados": duplicados_exactos,
        "duplicados_id_eliminados": duplicados_id,
        "edades_invalidas_convertidas_a_nulo": edades_invalidas,
        "filas_salida": len(datos)
    }

    return datos, auditoria


In [11]:
# Comparar con el ETL secuencial

resultado_secuencial, auditoria_secuencial = transformar_datos(
    df_original
)

print("Dimensiones secuenciales:", resultado_secuencial.shape)
print("Dimensiones paralelas:", df_paralelo.shape)

print(
    "¿Los resultados son idénticos?:",
    resultado_secuencial.equals(df_paralelo)
)

Dimensiones secuenciales: (108733, 42)
Dimensiones paralelas: (108733, 42)
¿Los resultados son idénticos?: True


### 5. Comparación de rendimiento

In [12]:
# 5. Comparación de rendimiento

repeticiones = 5
tiempos_secuenciales = []
tiempos_paralelos = []

for i in range(repeticiones):

    # Medir secuencial
    inicio = perf_counter()
    resultado_seq, _ = transformar_datos(df_original)
    tiempo_seq = perf_counter() - inicio
    tiempos_secuenciales.append(tiempo_seq)

    # Medir paralelo
    inicio = perf_counter()
    resultado_par, _ = ejecutar_etl_paralelo(
        df_original,
        numero_procesos=4
    )
    tiempo_par = perf_counter() - inicio
    tiempos_paralelos.append(tiempo_par)

    print(f"Repetición {i+1}:")
    print(f"  Secuencial: {tiempo_seq:.4f} segundos")
    print(f"  Paralelo:   {tiempo_par:.4f} segundos")

media_seq = np.mean(tiempos_secuenciales)
media_par = np.mean(tiempos_paralelos)

speedup = media_seq / media_par

print("\n--- RESULTADOS FINALES ---")
print(f"Media secuencial: {media_seq:.4f} segundos")
print(f"Media paralela:   {media_par:.4f} segundos")
print(f"Speedup:          {speedup:.4f}x")

Repetición 1:
  Secuencial: 0.8977 segundos
  Paralelo:   0.9211 segundos
Repetición 2:
  Secuencial: 0.8605 segundos
  Paralelo:   1.4798 segundos
Repetición 3:
  Secuencial: 1.0933 segundos
  Paralelo:   1.2746 segundos
Repetición 4:
  Secuencial: 0.8875 segundos
  Paralelo:   1.1423 segundos
Repetición 5:
  Secuencial: 0.5450 segundos
  Paralelo:   0.9685 segundos

--- RESULTADOS FINALES ---
Media secuencial: 0.8568 segundos
Media paralela:   1.1573 segundos
Speedup:          0.7404x


En las pruebas realizadas sobre un conjunto de datos de 108,733 registros, el procesamiento secuencial obtuvo un tiempo promedio de 0.8568 segundos, mientras que el procesamiento paralelo con cuatro procesos obtuvo un promedio de 1.1573 segundos. Se obtuvo un speedup de 0.7404x, lo que indica que, para el tamaño de datos evaluado, la ejecución paralela presentó un mayor tiempo de procesamiento. Esto puede atribuirse a la sobrecarga asociada con la creación de procesos, la distribución de los datos y la recopilación de resultados.

In [13]:
from time import perf_counter
import numpy as np

configuraciones = [2, 4, 8]
repeticiones = 5

resultados = []

for procesos in configuraciones:
    tiempos = []

    for i in range(repeticiones):
        inicio = perf_counter()

        resultado, auditoria = ejecutar_etl_paralelo(
            df_original,
            numero_procesos=procesos
        )

        tiempo = perf_counter() - inicio
        tiempos.append(tiempo)

    promedio = np.mean(tiempos)

    resultados.append({
        "Procesos": procesos,
        "Tiempo promedio": promedio,
        "Speedup": media_seq / promedio
    })

    print(
        f"{procesos} procesos: "
        f"{promedio:.4f} segundos | "
        f"Speedup: {media_seq / promedio:.4f}x"
    )

2 procesos: 1.2886 segundos | Speedup: 0.6649x
4 procesos: 1.0015 segundos | Speedup: 0.8555x
8 procesos: 1.1618 segundos | Speedup: 0.7375x
